# PULPO Uncertainty Package Toy Notebook: `pulpo_unc`

This notebook is a standalone companion to `pulpo_showcase.ipynb`, focused entirely on **`pulpo_unc`** — PULPO's curated uncertainty-data pipeline built on top of the core optimizer.

The main showcase notebook demonstrates uncertainty through `PulpoOptimizer.solve_MC`, which re-samples the raw Brightway LCI matrices (technosphere `A`, biosphere `B`, characterisation `Q`) on every iteration and re-solves the optimization. That works out-of-the-box but is a *black box*: every exchange in the databases is perturbed according to whatever distributions Brightway happens to carry, and there is no easy way to inspect, filter, or override them.

`pulpo.pulpo_unc` exposes a more controlled workflow built on top of the same optimizer. Its entry point `PulpoOptimizerUnc` (a subclass of `PulpoOptimizer`) adds a curated **uncertainty data pipeline** with six steps:

| Step | Method | Purpose |
|---|---|---|
| 1 | `import_and_filter_uncertainty_data` | Pull `stats_arrays` metadata and keep only parameters that matter (impact cutoff) |
| 2 | `apply_uncertainty_strategies` | Fill gaps / override distributions in an auditable way |
| 3 | `run_mc_from_uncertainty` | Monte Carlo on the curated distributions (no Brightway resample) |
| 4 | `create_CC_formulation` + `solve_CC_problem` | Chance-constrained optimization with a conservative L1 (Gaussian) approximation |
| 5 | `create_SOC_formulation` + `solve_SOC_problem` | Chance-constrained optimization with the *exact* second-order-cone formulation, including the covariance the L1 approximation misses |
| 6 | `run_gsa` | Global sensitivity analysis (SALib) to rank drivers of output variance |

**When to use which?** Use `solve_MC` (see `pulpo_showcase.ipynb`) for a quick, full-database robustness check. Use `pulpo_unc` when you need transparency over *which* parameters are uncertain, want to inject expert knowledge, or need derived analyses such as GSA or chance constraints. Within the chance-constrained workflow, prefer the exact SOC formulation (step 5) over the L1 shortcut (step 4) whenever the extra solve cost is affordable — the L1 approximation is conservative by an unquantified amount and can change the reported optimum, not just its precision.

> **Installation**: requires the extras `pip install "pulpo-dev[uncertainty]"` (pulls in `SALib`, `seaborn`, `matplotlib`, `stats_arrays`).

The sample database used throughout carries `NormalUncertainty` on its exchanges, so we can exercise the full workflow on the methanol + ozone system.

## 1. Sample Database Setup

In [ ]:
from pulpo import pulpo
pulpo.setup_sample_db()

## 2. Imports and Configuration

In [ ]:
import os
import numpy as np
np.NaN = np.nan
import pandas as pd
import matplotlib.pyplot as plt

from pulpo.utils.utils import is_bw25

project = "sample_project_bw25" if is_bw25() else "sample_project"
database = ["background_db", "foreground_db"]

# Single-method config (required by import_and_filter_uncertainty_data)
method_unc = {"('my project', 'climate change')": 1}

notebook_dir = os.path.dirname(os.getcwd())
directory = os.path.join(notebook_dir, 'data')

## 3. Instantiating `PulpoOptimizerUnc`

The constructor matches `PulpoOptimizer`. We restrict ourselves to a single LCIA method here — the current uncertainty-import routines operate on one method at a time.

In [ ]:
from pulpo import pulpo_unc
from pulpo.utils.uncertainty import processor

pulpo_worker_unc2 = pulpo_unc.PulpoOptimizerUnc(project, database, method_unc, directory)
pulpo_worker_unc2.get_lci_data()

## 4. System Definition (Technology Choices)

We exercise the workflow on the same methanol + ozone system used in `pulpo_showcase.ipynb`: alternative electricity (wind vs. natural gas), hydrogen (SMR vs. electrolysis), and oxygen (market vs. ASU, plus the electrolysis O2-byproduct) supply options.

In [ ]:
methanol_process = pulpo_worker_unc2.retrieve_processes(reference_products='methanol')
ozone_process = pulpo_worker_unc2.retrieve_processes(reference_products='ozone')

electricity_processes = pulpo_worker_unc2.retrieve_processes(
    processes=["wind electricity", "natural gas electricity"])
hydrogen_processes = pulpo_worker_unc2.retrieve_processes(
    processes=["hydrogen SMR", "hydrogen electrolysis"])
oxygen_processes = pulpo_worker_unc2.retrieve_processes(
    processes=["O2-market", "O2 ASU"])
oxygen_byproduct = pulpo_worker_unc2.retrieve_processes(processes=["O2-byproduct"])

In [ ]:
demand_unc2 = {methanol_process[0]: 1, ozone_process[0]: 2}
choices_unc2 = {
    "Electricity": {electricity_processes[0]: 1e10, electricity_processes[1]: 1e10},
    "Hydrogen":    {hydrogen_processes[0]: 1e10, hydrogen_processes[1]: 1e10},
    "Oxygen":      {oxygen_processes[0]: 1e10,   oxygen_processes[1]: 1e10},
}
lower_bound_unc2 = {oxygen_byproduct[0]: 0}

pulpo_worker_unc2.instantiate(
    choices=choices_unc2,
    demand=demand_unc2,
    lower_limit=lower_bound_unc2,
)
print("PulpoOptimizerUnc instantiated on the methanol + ozone system.")

## 5. Deterministic Reference Solve

In [ ]:
# Deterministic solve — provides the reference operating point for run_gsa.
pulpo_worker_unc2.solve()
det_result_data = pulpo_worker_unc2.extract_results()
print("Deterministic optimum (climate change):",
      det_result_data["Impacts"].loc["('my project', 'climate change')", "Value"])

## 6. Importing and Filtering Uncertainty Data

`import_and_filter_uncertainty_data` collects the `stats_arrays` metadata of every exchange involved and keeps only those whose contribution to the deterministic impact exceeds `cutoff` (fraction of total). This is the key difference with `solve_MC`: instead of resampling *everything*, we focus MC / GSA / CC on the parameters that actually move the objective.

The result is exposed as `pulpo_worker_unc2.uncertainty_data`, a nested dict split into three sections:
- `If` — intervention-flow parameters, grouped per database (`background_db`, `foreground_db`).
- `Cf` — characterisation-factor parameters, grouped per LCIA method.
- `Var_bounds` — decision-variable bounds (empty here, used when `choices` carry uncertainty).

Within each group, entries are split into **defined** (distribution already known) and **undefined** (need a gap-filling strategy in the next step).

In [ ]:
# Keep parameters contributing more than 0.1% to the deterministic impact.
# The default 'naive' strategy would require a prior solve (result_data).
# We use 'constructed_demand' instead, which scales every alternative in
# `choices` by 1 so all technology options are considered during filtering.
pulpo_worker_unc2.import_and_filter_uncertainty_data(
    cutoff=0.001,
    scaling_vector_strategy='constructed_demand',
)

# Inspect what was collected
unc_data = pulpo_worker_unc2.uncertainty_data
print("Uncertainty data sections:", list(unc_data.keys()))
for section in ("If", "Cf", "Var_bounds"):
    if section not in unc_data:
        continue
    for subgroup, entries in unc_data[section].items():
        print(f"  {section}[{subgroup}]: "
              f"{len(entries.get('defined', {}))} defined, "
              f"{len(entries.get('undefined', {}))} undefined")

## 7. Applying Gap-Filling Strategies

Real databases never carry distributions on every exchange. `apply_uncertainty_strategies` walks the `undefined` entries and synthesises a distribution for each, according to a list of user-supplied strategies.

- Passing `strategies=[]` activates the **built-in base case** — `TriangularBoundInterpolationStrategy` for `If` and a relative-scaling strategy for `Cf`. This is robust on full databases but needs several already-defined bounds per subgroup to interpolate from.
- Passing explicit strategies gives you full control. Below we use `TriangluarBaseStrategy` (simple ±x% triangular around the mean), which works on small demos like this sample database. For epistemic adjustments, `ExpertKnowledgeStrategy` lets you override individual parameters.

Setting `drop_undefined=True` removes anything still missing after the strategies have run, so downstream MC / GSA / CC only operate on fully specified parameters.

In [ ]:
# Build explicit gap-filling strategies. We bypass `strategies=[]` (the default
# base case) because it uses TriangularBoundInterpolationStrategy for intervention
# flows, which needs a broad population of already-defined bounds to interpolate
# from - our small sample database doesn't have enough to support that.
from pulpo.utils.uncertainty.processor import TriangluarBaseStrategy

db_names = pulpo_worker_unc2.database if isinstance(
    pulpo_worker_unc2.database, list
) else [pulpo_worker_unc2.database]
method_name = next(iter(pulpo_worker_unc2.method))

explicit_strategies = []
# Intervention flows: one strategy per background/foreground database
for db in db_names:
    if db in pulpo_worker_unc2.uncertainty_data.get('If', {}):
        explicit_strategies.append(TriangluarBaseStrategy(
            uncertain_param_type='If',
            uncertain_param_subgroup=db,
            upper_scaling_factor=0.1,     # 10% around the mean
            lower_scaling_factor=0.1,
            noise_interval={'min': 0.1, 'max': 0.1},
        ))
# Characterisation factors for the active LCIA method
if method_name in pulpo_worker_unc2.uncertainty_data.get('Cf', {}):
    explicit_strategies.append(TriangluarBaseStrategy(
        uncertain_param_type='Cf',
        uncertain_param_subgroup=method_name,
        upper_scaling_factor=0.05,        # 5% around the mean
        lower_scaling_factor=0.05,
        noise_interval={'min': 0.1, 'max': 0.1},
    ))

pulpo_worker_unc2.apply_uncertainty_strategies(
    strategies=explicit_strategies,
    drop_undefined=True,   # discard anything still missing after the strategies
)
print("Uncertainty strategies applied; remaining undefined entries dropped.")

## 8. Monte Carlo from Prepared Distributions

`run_mc_from_uncertainty` is the `pulpo_unc` counterpart to `solve_MC` (see `pulpo_showcase.ipynb`). The two differ in **what** and **how** they sample:

| | `solve_MC` (`pulpo_showcase.ipynb`) | `run_mc_from_uncertainty` (here) |
|---|---|---|
| Source of randomness | Brightway resample of full `A`, `B`, `Q` matrices | The curated `uncertainty_data` dict built in Sections 6-7 |
| Parameters sampled | Every uncertain exchange in the databases | Only the subset surviving the cutoff + strategies |
| Distributions | Whatever `stats_arrays` is attached in the database | The (possibly overridden) distributions from Section 7 |
| Speed | Slower — re-samples and re-builds matrices each iteration | Faster — draws samples up front and patches a small overlay |

In short: `solve_MC` answers *"how uncertain is my full system?"*, `run_mc_from_uncertainty` answers *"how uncertain is my system given this specific set of distributions?"*.

In [ ]:
%%capture
mc_unc_results = pulpo_worker_unc2.run_mc_from_uncertainty(
    n_samples=200,
    seed=42,
    n_jobs=1,  # single process keeps output readable in a notebook; use -1 to parallelize
)

# Extract the climate-change impact per iteration
cc_key = "('my project', 'climate change')"
cc_samples = []
for it, res in mc_unc_results.items():
    if isinstance(res, dict) and "error" in res:
        continue
    try:
        cc_samples.append(res["Impacts"].loc[cc_key, "Value"])
    except Exception:
        pass

cc_samples = np.asarray(cc_samples, dtype=float)
print(f"Successful MC iterations: {len(cc_samples)} / {len(mc_unc_results)}")
print(f"Climate change   mean = {cc_samples.mean():.4f}")
print(f"Climate change    std = {cc_samples.std():.4f}")
print(f"Climate change 5-95% = [{np.quantile(cc_samples, 0.05):.4f}, "
      f"{np.quantile(cc_samples, 0.95):.4f}]")

In [ ]:
plt.figure(figsize=(6, 4))
plt.hist(cc_samples, bins=30, color="#4C72B0", alpha=0.8, edgecolor="white")
plt.axvline(cc_samples.mean(), color="red", linestyle="--",
            label=f"mean = {cc_samples.mean():.4f}")
plt.xlabel("Climate change impact")
plt.ylabel("Frequency")
plt.title("MC from prepared uncertainty distributions")
plt.legend()
plt.tight_layout()
plt.show()

## 9. Chance-Constrained Optimization

The CC workflow reformulates the optimization so the expected environmental cost is satisfied with probability `λ`, under a Gaussian approximation of the aggregated impact:

1. `create_CC_formulation` fits the required mean / variance structure from the curated uncertainty data.
2. `solve_CC_problem` plugs that structure into the Pyomo model and solves one or several `λ` levels, tracing the Pareto front between optimality and robustness.

In [ ]:
# 1) Build the CC statistics from the prepared uncertainty data
normal_metadata_env_cost, normal_metadata_var_bounds = pulpo_worker_unc2.create_CC_formulation(
    CC_env_cost=True,
    CC_var_bounds=[],                    # keep it simple: only env-cost CC here
    normal_transformation_sample_size=100,
)

# 2) Solve the CC problem for a handful of confidence levels
lambda_levels = [0.50, 0.75, 0.90, 0.95]
results_CC = pulpo_worker_unc2.solve_CC_problem(
    lambda_level=lambda_levels,
    normal_metadata_env_cost=normal_metadata_env_cost,
    normal_metadata_var_bounds=normal_metadata_var_bounds,
    plot_results=False,
)

# 3) Summarize the climate-change impact at each λ
summary = []
for lam, rd in results_CC.items():
    impact = rd["Impacts"].loc[cc_key, "Value"]
    summary.append((float(lam), impact))

cc_summary_df = pd.DataFrame(summary, columns=["lambda", "climate_change_impact"])
cc_summary_df

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(cc_summary_df["lambda"], cc_summary_df["climate_change_impact"],
         "o-", color="#C44E52", linewidth=2)
plt.xlabel("Confidence level λ")
plt.ylabel("Climate change impact at optimum")
plt.title("Chance-constrained Pareto trace (env-cost only)")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 10. Exact (Second-Order-Cone) Chance-Constrained Optimization

Section 9's `create_CC_formulation` / `solve_CC_problem` uses a *shortcut*: it builds a **per-process** standard deviation

$$\sigma_j = \sqrt{\sum_e \left(\mu_{q,e}^2 \sigma_{b,ej}^2 + \mu_{b,ej}^2 \sigma_{q,e}^2 + \sigma_{q,e}^2 \sigma_{b,ej}^2\right)}$$

and aggregates it across processes with an **L1 sum**, $\sum_j s_j (\mu_j + z_\lambda \sigma_j)$. That has two consequences:

1. the L1 norm over-estimates the true (Euclidean, L2) norm of the uncertainty — it is conservative by an amount that is never quantified;
2. more importantly, the **same** characterization factor $q_e$ multiplies every process $j$ that emits flow $e$, so their contributions to the impact are correlated. Summing per-process *variances* silently assumes they are independent.

Both problems are fixed by aggregating the biosphere row **before** characterizing it. Writing the life-cycle impact as $X = \sum_e q_e g_e$ with $g_e = \sum_j b_{ej} s_j$ (keeping the same independence assumptions the L1 derivation already makes: $q$ independent of $b$, entries of $b$ mutually independent, $q_e$ independent across $e$):

$$
\mathrm{Var}[X] = \underbrace{\sum_j d_j\, s_j^2}_{\text{per-process}} \;+\; \underbrace{\sum_e w_e\, y_e^2}_{\text{shared-CF covariance}}, \qquad
d_j = \sum_e (\mu_{q,e}^2+\sigma_{q,e}^2)\,\sigma_{b,ej}^2, \quad w_e = \sigma_{q,e}^2, \quad y_e = \sum_j \mu_{b,ej}\, s_j
$$

which is a *diagonal quadratic form* in the scaling vector $s$ — exact, and with the covariance term restored. The deterministic-equivalent problem

$$
\min\; \mu^\top s + \Phi^{-1}(\lambda)\, t \qquad \text{s.t.} \qquad \sum_j d_j s_j^2 + \sum_e w_e y_e^2 \le t^2,\;\; t \ge 0
$$

is a convex **second-order cone program (SOCP)** with one extra variable plus one auxiliary variable per *characterized* flow — not per process, so it stays small even at ecoinvent scale.

**Why not always use it?** Handing the cone straight to an interior-point solver is only reliable on small, well-scaled systems. On ecoinvent-scale technosphere matrices the coefficient range (`d_j` can span ~20 orders of magnitude) breaks the barrier method's conditioning. PULPO offers two solve strategies behind one API:

| `method=` | How | When to use |
|---|---|---|
| `'cutting_plane'` (default) | Kelley cutting planes: $\sigma(s)$ is convex and positively homogeneous, hence the supremum of its supporting hyperplanes, so every iteration solves the *same LP* PULPO already solves elsewhere, plus one more linear cut. Converges to a certified optimality gap in a handful of iterations. | Any scale, any of PULPO's solvers (HiGHS, Gurobi, ...) |
| `'direct'` | Hands the cone to Gurobi as a QCP, one solve per $\lambda$, no cuts. | Small/well-scaled systems only (like this toy example); requires a working `"gurobi"` Pyomo solver |

The one assumption *not* repaired here is correlation among the `b_ej` themselves — ecoinvent stores none, so it cannot be modeled. That biases the variance slightly downward (the chance constraint stays a little optimistic in that one respect); everything else about the L1 shortcut is corrected.

The two-step API mirrors Section 9's: `create_SOC_formulation` assembles the coefficients ($\mu_j$, $d_j$, $w_e$, ...) from the same curated `uncertainty_data`, and `solve_SOC_problem` plugs them into the model and sweeps $\lambda$. See `pulpo/utils/uncertainty/soc.py` for the full derivation and numerical notes.

In [ ]:
# 1) Assemble the exact-variance coefficients from the same curated uncertainty data
soc_coeffs = pulpo_worker_unc2.create_SOC_formulation(
    normal_transformation_sample_size=100,
)
soc_coeffs.summary()

In [ ]:
# 2) Solve the exact SOC problem at the same lambda levels as the L1 CC trace above.
# The cutting-plane method reuses the plain LP solve already used in Section 9,
# so no extra solver (e.g. Gurobi) is required.
results_SOC = pulpo_worker_unc2.solve_SOC_problem(
    lambda_level=lambda_levels,
    coeffs=soc_coeffs,
    normal_metadata_var_bounds=normal_metadata_var_bounds,
    method='cutting_plane',
    plot_results=False,
)

soc_summary = []
for lam, rd in results_SOC.items():
    impact = rd["Impacts"].loc[cc_key, "Value"]
    soc_summary.append((float(lam), impact))

soc_summary_df = pd.DataFrame(soc_summary, columns=["lambda", "climate_change_impact"])
soc_summary_df

In [ ]:
# 3) Compare against the L1 trace from Section 9. For a correct implementation the
# L1 front can only lie at or above the exact one (L1 is conservative, never optimistic).
comparison_df = cc_summary_df.merge(
    soc_summary_df, on="lambda", suffixes=("_L1", "_exact_SOC")
)
comparison_df["L1_minus_exact"] = (
    comparison_df["climate_change_impact_L1"] - comparison_df["climate_change_impact_exact_SOC"]
)
comparison_df

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(comparison_df["lambda"], comparison_df["climate_change_impact_L1"],
         "o-", color="#C44E52", linewidth=2, label="L1 shortcut (Section 9)")
plt.plot(comparison_df["lambda"], comparison_df["climate_change_impact_exact_SOC"],
         "s--", color="#4C72B0", linewidth=2, label="exact SOC (this section)")
plt.xlabel("Confidence level λ")
plt.ylabel("Climate change impact at optimum")
plt.title("L1 shortcut vs. exact second-order-cone chance constraint")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

On this small toy database the two fronts are close, because few processes here share an uncertain characterization factor — the shared-CF covariance term the L1 shortcut misses is small in this example. On ecoinvent-scale systems the gap is not small: in the case study that motivated this formulation, the L1 front overstated the impact by several megatonnes CO2-eq/yr at every λ above 0.5, and the approximation didn't just cost accuracy — it changed which technology the optimizer picked at 10 of 17 Pareto points. See `pulpo/utils/uncertainty/soc.py` for the full derivation and numerical notes on why an ecoinvent-scale problem needs the cutting-plane solve.

Restore the plain deterministic objective before moving on. Section 11's GSA doesn't touch the live Pyomo objective (it re-evaluates impacts algebraically at a fixed scaling vector), but leaving the SOC cone/cuts in place would silently affect any later `.solve()` call on this instance.

In [ ]:
pulpo_worker_unc2.restore_deterministic_objective()
print("Objective restored to the plain weighted-impact sum.")

## 11. Global Sensitivity Analysis

`run_gsa` wraps SALib to rank parameters by their contribution to the output variance. It reuses the curated `uncertainty_data` and anchors the scaling vector in a deterministic `result_data` (obtained via `extract_results()`).

The raw output uses internal matrix indices as parameter names (tuples `(intervention_idx, process_idx)` for `If` and a single `intervention_idx` for `Cf`). The cell below joins those against `lci_data['process_map_metadata']` and `lci_data['intervention_map_metadata']` to produce human-readable labels of the form *"process → flow"* (or *"flow → LCIA method"* for characterisation factors).

The Sobol sample size is set very low for demo speed; increase `sample_size` for production studies.

In [ ]:
# Global sensitivity analysis (Sobol) on the uncertain parameters.
# `run_gsa` returns:
#   - total_order (pd.DataFrame): columns ST / ST_conf indexed by parameter name
#   - sensitivity_indices (SALib ResultDict): the full SALib analysis result
from SALib.sample import sobol as sobol_sample
from SALib.analyze import sobol as sobol_analyze

gsa_total_order, sensitivity_indices = pulpo_worker_unc2.run_gsa(
    result_data=det_result_data,
    sample_method=sobol_sample,
    SA_method=sobol_analyze,
    sample_size=32,          # tiny, just for the showcase
    plot_gsa_results=False,
    top_sensitivity_amt=10,
)

# Raw indices -> human-readable names using the lci_data metadata maps.
process_map = pulpo_worker_unc2.lci_data['process_map_metadata']
interv_map  = pulpo_worker_unc2.lci_data['intervention_map_metadata']
method_name = next(iter(pulpo_worker_unc2.method))

def _label(param):
    # If-parameters are (intervention_idx, process_idx) tuples;
    # Cf-parameters are a bare intervention_idx.
    if isinstance(param, tuple):
        interv_idx, proc_idx = param
        return f"{process_map.get(proc_idx, proc_idx)}  ←  {interv_map.get(interv_idx, interv_idx)}"
    return f"{interv_map.get(param, param)}  (CF: {method_name})"

gsa_labeled = gsa_total_order.copy()
gsa_labeled.index = [_label(p) for p in gsa_labeled.index]
gsa_labeled.index.name = "parameter"

print("Top parameters by total-order Sobol index:")
gsa_labeled.sort_values('ST', ascending=False).head(10)

## 12. Summary

We just ran the full `pulpo_unc` workflow on the sample system:

1. **Import & filter** (Section 6) — kept only parameters contributing >0.1% to the deterministic impact.
2. **Gap-filling strategies** (Section 7) — assigned explicit ±10% / ±5% triangular distributions.
3. **Monte Carlo** (Section 8) — 200 optimization runs on the curated distributions, producing an empirical impact distribution.
4. **Chance constraints, L1 shortcut** (Section 9) — traced the impact-robustness frontier for λ ∈ {0.50, 0.75, 0.90, 0.95} using a conservative per-process L1 aggregation.
5. **Chance constraints, exact SOC** (Section 10) — traced the same frontier with the exact second-order-cone formulation (including the shared-characterization-factor covariance the L1 shortcut misses), and confirmed it never lies above the L1 front.
6. **Global sensitivity** (Section 11) — ranked parameters by Sobol total-order index.

Compared to the black-box `solve_MC` demonstrated in `pulpo_showcase.ipynb`, this workflow gives you a paper trail for every uncertain parameter and unlocks downstream analyses (GSA, CC, exact SOC) that require structured distribution information. Use it for epistemic studies (e.g. the ammonia showcase) where *which* parameter drives the variance is as important as the variance itself, and reach for the exact SOC formulation over the L1 shortcut whenever the extra solve cost (roughly 2-3x, via cutting planes) is affordable — it is not merely more accurate, it can change the reported optimum.